In [ ]:
import numpy as np
from numpy.linalg import inv
import time


class Position:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)


def transform_goal(goal_pos, pos, psi):
    # Create the homogeneous transformation from robot to inertial frame
    R_r2i = np.array(
        [
            [np.cos(psi), -np.sin(psi), pos.x],
            [np.sin(psi), np.cos(psi), pos.y],
            [0, 0, 1.0],
        ]
    )
    # Invert to get the inertial to robot frame transformation
    R_i2r = inv(R_r2i)
    # Represent the goal in homogeneous coordinates
    pi = np.array([goal_pos[0], goal_pos[1], 1.0])
    # Transform the goal and return the first two coordinates
    pr = R_i2r @ pi
    return [pr[0], pr[1]]


def transform_goal_alt(goal_pos, pos, psi):
    dx = goal_pos[0] - pos.x
    dy = goal_pos[1] - pos.y
    # Directly compute the transformation
    return [np.cos(psi) * dx + np.sin(psi) * dy, -np.sin(psi) * dx + np.cos(psi) * dy]


def transform_goal_matmul(goal_pos, pos, psi):
    # Create the homogeneous transformation from robot to inertial frame
    cos_psi = np.cos(psi)
    sin_psi = np.sin(psi)
    R_w2r = np.array(
        [
            [cos_psi, sin_psi, -pos.x * cos_psi - pos.y * sin_psi],
            [-sin_psi, cos_psi, pos.x * sin_psi - pos.y * cos_psi],
            [0, 0, 1.0],
        ]
    )
    # Represent the goal in homogeneous coordinates
    pi = np.array([goal_pos[0], goal_pos[1], 1.0])
    # Transform the goal and return the first two coordinates
    pr = R_w2r @ pi
    return [pr[0], pr[1]]


def transform_goal_optimized(goal_pos, pos, psi):
    # Calculate sin and cos only once
    cos_psi = np.cos(psi)
    sin_psi = np.sin(psi)

    # Direct calculation with minimal operations
    dx = goal_pos[0] - pos.x
    dy = goal_pos[1] - pos.y

    return [cos_psi * dx + sin_psi * dy, -sin_psi * dx + cos_psi * dy]


# Test parameters:
pos = Position(1.0, 2.0)
psi = np.pi / 3
goal = [10.0, 10.0]

# Compute execution times
num_iterations = 100000
start_time = time.time()
for _ in range(num_iterations):
    res1 = transform_goal(goal, pos, psi)
end_time = time.time()
time_transform_goal = (end_time - start_time) / num_iterations

start_time = time.time()
for _ in range(num_iterations):
    res2 = transform_goal_alt(goal, pos, psi)
end_time = time.time()
time_transform_goal_alt = (end_time - start_time) / num_iterations

start_time = time.time()
for _ in range(num_iterations):
    res3 = transform_goal_optimized(goal, pos, psi)
end_time = time.time()
time_transform_goal_matmul = (end_time - start_time) / num_iterations

print("transform_goal:", res1)
print("transform_goal_alt:", res2)
print("transform_goal_matmul:", res3)
print(f"Execution time of transform_goal: {time_transform_goal * 1e6:.2f} microseconds")
print(
    f"Execution time of transform_goal_alt: {time_transform_goal_alt * 1e6:.2f} microseconds"
)
print(
    f"Execution time of transform_goal_matmul: {time_transform_goal_matmul * 1e6:.2f} microseconds"
)
print(f"Speed improvement: {time_transform_goal / time_transform_goal_alt:.2f}x")
print(
    f"Speed improvement (matmul): {time_transform_goal / time_transform_goal_matmul:.2f}x"
)

transform_goal: [11.428203230275509, -3.7942286340599463]
transform_goal_alt: [11.42820323027551, -3.7942286340599463]
transform_goal_matmul: [11.42820323027551, -3.7942286340599463]
Execution time of transform_goal: 8.66 microseconds
Execution time of transform_goal_alt: 2.22 microseconds
Execution time of transform_goal_matmul: 1.30 microseconds
Speed improvement: 3.90x
Speed improvement (matmul): 6.65x


In [ ]:
import time
import numpy as np


# Create test versions with different names
def obs_dist_reward_original(laser_scan, alpha=0.1):
    laser_data = laser_scan
    valid_ranges = [r for r in laser_data.ranges if r >= 0 and r != float("inf")]
    min_distance = min(valid_ranges)  # if valid_ranges else float('inf')
    return min(0.01, alpha * min_distance)


def obs_dist_reward_numpy(laser_scan, alpha=0.1):
    laser_data = laser_scan
    ranges = np.array(laser_data.ranges)
    valid = (ranges > 0) & (ranges != np.inf)
    if np.any(valid):
        min_distance = np.min(ranges[valid])
    else:
        min_distance = float("inf")
    return min(0.01, alpha * min_distance)


# Create a mock LaserScan class to test the functions
class MockLaserScan:
    def __init__(self, ranges):
        self.ranges = ranges


# Test data for validation
test_laser_scans = [
    MockLaserScan([0.5, 1.2, 0.3, 0.8, 2.0]),  # Normal case
    MockLaserScan([0.0, 1.2, 0.3, 0.8, 2.0]),  # Zero value
    MockLaserScan([float("inf"), 1.2, 0.3, 0.8, 2.0]),  # Infinity value
    MockLaserScan([0.001, float("inf"), 0.3, 0.8, 2.0]),  # Mix of zero and infinity
    MockLaserScan([float("inf"), float("inf")]),  # All infinity
    MockLaserScan([0.0, 0.0]),  # All zeros
]

# Check if both functions return the same results
print("Validation test:")
for i, scan in enumerate(test_laser_scans):
    result_numpy = obs_dist_reward_numpy(scan)
    print(f"Test case {i}: NumPy: {result_numpy}")

# Create a larger dataset for benchmarking
# Generate 1000 random values for large_ranges - mix of normal values, zeros and infinities
np.random.seed(42)
large_ranges = np.random.rand(1000) * 5  # Random values between 0 and 5
# Add some zeros and inf values to simulate laser scan behavior
zero_indices = np.random.choice(1000, 50, replace=False)
inf_indices = np.random.choice(1000, 50, replace=False)
large_ranges[zero_indices] = 0.0
large_ranges[inf_indices] = float("inf")
large_ranges = large_ranges.tolist()  # Convert to list for consistency

large_scan = MockLaserScan(large_ranges)

# Benchmark performance
num_runs = 10000
print("\nPerformance test:")

start_time = time.time()
for _ in range(num_runs):
    result_original = obs_dist_reward_original(large_scan)
end_time = time.time()
original_time = (end_time - start_time) / num_runs

start_time = time.time()
for _ in range(num_runs):
    result_numpy = obs_dist_reward_numpy(large_scan)
end_time = time.time()
numpy_time = (end_time - start_time) / num_runs

print(f"Original implementation: {original_time * 1e6:.2f} microseconds")
print(f"NumPy implementation: {numpy_time * 1e6:.2f} microseconds")
print(f"Speed improvement: {original_time / numpy_time:.2f}x")

Validation test:
Test case 0: NumPy: 0.01
Test case 1: NumPy: 0.01
Test case 2: NumPy: 0.01
Test case 3: NumPy: 0.0001
Test case 4: NumPy: 0.01
Test case 5: NumPy: 0.01

Performance test:
Original implementation: 79.92 microseconds
NumPy implementation: 36.97 microseconds
Speed improvement: 2.16x


In [ ]:
from gymnasium import spaces
import gymnasium as gym
import numpy as np

import matplotlib.pyplot as plt


class SimpleGridWorld(gym.Env):
    """
    A simple 2D grid world where the agent needs to navigate to a goal position.

    Actions:
    0: Move up
    1: Move right
    2: Move down
    3: Move left

    Observations:
    Agent's x, y position and goal's x, y position

    Reward:
    1.0 if reached goal, -0.1 for each step, -1.0 if hitting a wall
    """

    def __init__(self, grid_size=10):
        super(SimpleGridWorld, self).__init__()

        self.grid_size = grid_size

        # Action space: up, right, down, left
        self.action_space = spaces.Discrete(4)

        # Observation space: [agent_x, agent_y, goal_x, goal_y]
        self.observation_space = spaces.Box(
            low=0, high=grid_size - 1, shape=(4,), dtype=np.float32
        )

        # Initialize variables
        self.agent_pos = None
        self.goal_pos = None
        self.reset()

    def reset(self, seed=1, options=None):
        # Place agent at random position
        self.agent_pos = np.array(
            [np.random.randint(0, self.grid_size), np.random.randint(0, self.grid_size)]
        )

        # Place goal at random position (different from agent)
        while True:
            self.goal_pos = np.array(
                [
                    np.random.randint(0, self.grid_size),
                    np.random.randint(0, self.grid_size),
                ]
            )
            # Ensure goal and agent are not at the same position
            if not np.array_equal(self.agent_pos, self.goal_pos):
                break

        return self._get_observation()

    def step(self, action):
        # Store previous position
        prev_pos = self.agent_pos.copy()

        # Move according to action
        if action == 0:  # up
            self.agent_pos[1] = min(self.agent_pos[1] + 1, self.grid_size - 1)
        elif action == 1:  # right
            self.agent_pos[0] = min(self.agent_pos[0] + 1, self.grid_size - 1)
        elif action == 2:  # down
            self.agent_pos[1] = max(self.agent_pos[1] - 1, 0)
        elif action == 3:  # left
            self.agent_pos[0] = max(self.agent_pos[0] - 1, 0)

        # Check if hit wall (didn't move)
        hit_wall = np.array_equal(prev_pos, self.agent_pos)

        # Check if reached goal
        reached_goal = np.array_equal(self.agent_pos, self.goal_pos)

        # Calculate reward
        if reached_goal:
            reward = 1.0
        elif hit_wall:
            reward = -1.0
        else:
            reward = -0.1

        # Get observation
        observation = self._get_observation()

        # Check if episode is done
        terminated = reached_goal
        
        # Check if episode is truncated
        truncated = False

        # Info dictionary
        info = {}

        return observation, reward, truncated, terminated, info

    def _get_observation(self):
        return np.concatenate([self.agent_pos, self.goal_pos]).astype(np.float32)

    def render(self, mode="human"):
        plt.figure(figsize=(5, 5))
        plt.xlim(0, self.grid_size)
        plt.ylim(0, self.grid_size)
        plt.grid(True)

        # Plot agent
        plt.scatter(
            self.agent_pos[0] + 0.5,
            self.agent_pos[1] + 0.5,
            color="blue",
            s=200,
            marker="o",
            label="Agent",
        )

        # Plot goal
        plt.scatter(
            self.goal_pos[0] + 0.5,
            self.goal_pos[1] + 0.5,
            color="green",
            s=200,
            marker="*",
            label="Goal",
        )

        plt.legend()
        plt.title("SimpleGridWorld")
        plt.xticks(np.arange(0, self.grid_size, 1))
        plt.yticks(np.arange(0, self.grid_size, 1))
        plt.show()


In [11]:
from gymnasium.wrappers import NormalizeReward

env = NormalizeReward(SimpleGridWorld())
env.reset()

env.step(env.action_space.sample())

ValueError: not enough values to unpack (expected 5, got 4)